# Blackberry 3DGS on Google Colab (`engines_cuda`)

Uses **PyTorch + gsplat** (CUDA). Branch: **`feature/colab-train`**.

**Runtime -> GPU** required (T4/L4/A100 — not TPU).

### Important (2026-10-28)
The first MCMC run (`cuda_out_mcmc`, `splat_07000.ply`) **diverged**: Gaussian means
flew to ±1e5 while opacity collapsed. Do **not** `--resume` that folder.

This notebook defaults to a **fresh OUT** and a **safer** recipe:
1. Warmup with `--strategy default`, or
2. MCMC with much lower `--noise-lr` (default 5e5 is too violent on this COLMAP scale).

Mesh: `engines_cuda/export/mesh_from_ckpt.py` (Open3D / PyTorch3D) — section 6.


## 1. GPU check


In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime type → GPU (not TPU)"
print(torch.__version__, torch.cuda.get_device_name(0), "VRAM_GiB",
      round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))


## 2. Clone repo + install deps

Pulls latest `feature/colab-train`. PyTorch3D needs `--no-build-isolation` after torch/gsplat.


In [ ]:
import os
REPO = "/content/splats"
BRANCH = "feature/colab-train"
if not os.path.isdir(REPO):
    !git clone --branch {BRANCH} --single-branch https://github.com/aidazsolt-bot/splats.git {REPO}
else:
    %cd {REPO}
    !git fetch origin {BRANCH} && git checkout {BRANCH} && git pull
%cd {REPO}
!pip -q install -U pip
!pip -q install -r engines_cuda/requirements.txt
!pip -q install --no-build-isolation -r engines_cuda/requirements-pytorch3d.txt
!python -c "from gsplat.cuda._backend import _C; print('gsplat CUDA ok', _C is not None)"


## 3. Mount Drive + dataset / dataset paths

Expected:
```
MyDrive/blackberry-splats/dataset/{cameras,images,points3D}.txt + images/
```

`OUT` must be a **new** folder after the failed MCMC run.


In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

DATA = "/content/drive/MyDrive/blackberry-splats/dataset"
# NEW out dir — do not reuse .../cuda_out_mcmc (diverged means)
OUT = "/content/drive/MyDrive/blackberry-splats/cuda_out_stable"

os.makedirs(os.path.dirname(DATA), exist_ok=True)
if not os.path.isfile(f"{DATA}/cameras.txt"):
    print("Dataset missing — downloading blackberry.zip once...")
    os.makedirs(DATA, exist_ok=True)
    !wget -O /content/blackberry.zip \
        "https://github.com/danybittel/splats/releases/download/berries/blackberry.zip"
    !unzip -q -o /content/blackberry.zip -d "{DATA}"
else:
    print("Dataset already on Drive — skip zip download")

assert os.path.isfile(f"{DATA}/cameras.txt"), f"Missing COLMAP at {DATA}"
assert os.path.isdir(f"{DATA}/images"), f"Missing images/ at {DATA}"
os.makedirs(OUT, exist_ok=True)
print("DATA", DATA)
print("OUT", OUT)
!ls "{DATA}" | head


## 4. Smoke train (optional)

Stack check only. Uses `default` + `data-factor 4`.


In [ ]:
%cd /content/splats
!python engines_cuda/run_train.py --smoke --data "{DATA}" --out "{OUT}/smoke" \
  --data-factor 4 --strategy default


## 5. Full train (stable recipe)

Failed run used MCMC + default `--noise-lr 5e5` → means exploded by step 7000.

| Mode | When |
|------|------|
| `default` | Safer first full train (recommended) |
| `mcmc` | Only with **low** `NOISE_LR` (e.g. `5e3`…`5e4`) and fresh `OUT` |

| GPU | `DATA_FACTOR` |
|-----|----------------|
| T4 | start `2`, try `1` if VRAM allows |
| L4 / A100 | `1` |

After a **healthy** disconnect: `RESUME = True` and **same** `OUT` + flags.
Never resume the old `cuda_out_mcmc`.


In [ ]:
%cd /content/splats

RESUME = False  # True only after a healthy disconnect on THIS OUT
STRATEGY = "default"  # or "mcmc" with low NOISE_LR
DATA_FACTOR = 2  # T4-safe; set 1 on L4/A100
CAP_MAX = 200000  # used by mcmc
MAX_STEPS = 30000  # raise to 45000 once stable
NOISE_LR = 5e3  # MCMC only; 5e5 diverged on this scene scale
ANTIALIASED = True

cmd = (
    f'python engines_cuda/run_train.py --data "{DATA}" --out "{OUT}" '
    f'--strategy {STRATEGY} --max-steps {MAX_STEPS} '
    f'--data-factor {DATA_FACTOR}'
)
if STRATEGY == "mcmc":
    cmd += f" --cap-max {CAP_MAX} --noise-lr {NOISE_LR}"
if ANTIALIASED:
    cmd += " --antialiased"
if RESUME:
    cmd += " --resume"
print(cmd)
!{cmd}


## 5b. Quick PLY sanity check (after first `splat_*.ply`)

Extent should be ~**20–70** per axis (COLMAP scale), not 1e5+.
Opacity median should not sit near ~0.01 with most mass outside the scene.


In [ ]:
import numpy as np
from pathlib import Path

ply = sorted(Path(OUT).glob("splat_*.ply"))
assert ply, f"No splat_*.ply yet in {OUT}"
path = ply[-1]
raw = path.read_bytes()
i = raw.find(b"end_header\n")
hdr, body = raw[:i].decode(), raw[i + 11 :]
n = int([ln for ln in hdr.splitlines() if ln.startswith("element vertex")][0].split()[-1])
props = [ln.split()[-1] for ln in hdr.splitlines() if ln.startswith("property float")]
arr = np.frombuffer(body, dtype="<f4", count=n * len(props)).reshape(n, len(props))
ix = {p: j for j, p in enumerate(props)}
xyz = arr[:, [ix["x"], ix["y"], ix["z"]]]
opa = 1 / (1 + np.exp(-np.clip(arr[:, ix["opacity"]], -40, 40)))
extent = xyz.max(0) - xyz.min(0)
print(path.name, "n", n)
print("extent", extent)
print("center", xyz.mean(0))
print("opacity p50/mean", float(np.median(opa)), float(opa.mean()))
if extent.max() > 500:
    print("FAIL: means likely diverged — stop and change STRATEGY/NOISE_LR/OUT")
else:
    print("OK: extent looks COLMAP-scale")


## 6. List outputs / mesh from checkpoint

Open3D Poisson (default) or PyTorch3D cubify.


In [ ]:
!ls -lh "{OUT}"
# !python engines_cuda/export/mesh_from_ckpt.py --ckpt "{OUT}/ckpt_30000.pt" --out "{OUT}/mesh_from_splat.ply"
# !python engines_cuda/export/mesh_from_ckpt.py --ckpt "{OUT}/ckpt_30000.pt" --out "{OUT}/mesh_from_splat_p3d.ply" \
#   --backend pytorch3d --grid-size 128
